In [ ]:
"""
TOSS inference API for notebooks — same sampling pipeline as app.py (no Gradio, no CLI).

Example:
    from inference import TossInference
    runner = TossInference(resume_path="ckpt/toss.ckpt")
    out = runner.generate("input.png", prompt="a red shoe", dy=-90)
    out = runner.generate(pil_image, prompt="", dx=0, dy=0, dz=0)
"""
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
from typing import Union

import numpy as np
import torch
from einops import rearrange
from omegaconf import OmegaConf
from PIL import Image
from pytorch_lightning import seed_everything
from torchvision import transforms

from ldm.models.diffusion.ddim import DDIMSampler
from ldm.util import instantiate_from_config

from cldm.toss import TOSS
from cldm.model import load_state_dict



from app import get_T_from_relative, preprocess_image, sample_model

def load_model(device, _hparams, sd_locked, only_mid_control, cfgs):
    import os
    os.environ["WANDB_MODE"] = "disabled"
    model = instantiate_from_config(cfgs.model)

    # Load the state dict
    state_dict = torch.load("/content/drive/MyDrive/checkpoints/toss.ckpt", map_location="cpu", weights_only=False)["state_dict"]

    # --- FIX START ---
    # Remove the problematic key if it exists
    key_to_remove = "cond_stage_model.transformer.text_model.embeddings.position_ids"
    if key_to_remove in state_dict:
        print(f"Removing {key_to_remove} from state_dict to match model structure.")
        del state_dict[key_to_remove]
    # --- FIX END ---

    model.load_state_dict(state_dict, strict=False) # Adding strict=False is a safe backup

    trained_ckpt = torch.load("/content/drive/MyDrive/checkpoints/toss_lora/toss-step=3000-v18.ckpt", map_location="cpu", weights_only=False)

    lora_keys = [k for k in trained_ckpt["state_dict"].keys() if "lora" in k.lower()]
    print(f"Found {len(lora_keys)} LoRA keys in checkpoint:")
    for k in lora_keys:
        print(f"  {k}: {trained_ckpt['state_dict'][k].abs().mean().item():.6f}")

    # This will inject your LoRA weights and your trained PoseNet
    m, u = model.load_state_dict(trained_ckpt["state_dict"], strict=False)

    print()

    for n, p in model.named_parameters():
      if "lora" in n.lower() or ".out." in n.lower():
          print(n, p.abs().mean().item())

    print("Missing:", m)
    print("Unexpected keys (should be empty):", u)

    # reweight noise scheduler
    if _hparams.register_scheduler:
        model.register_schedule(given_betas=None, beta_schedule="linear", timesteps=1000, linear_start=0.00085, linear_end=0.016)

    model.learning_rate = _hparams.lr
    model.sd_locked = sd_locked
    model.only_mid_control = only_mid_control
    model = model.to(device)
    model.eval()
    return model

ImageInput = Union[str, Path, Image.Image, np.ndarray]


def _to_pil_rgba(image: ImageInput) -> Image.Image:
    if isinstance(image, (str, Path)):
        return Image.open(image).convert("RGBA")
    if isinstance(image, Image.Image):
        return image.convert("RGBA")
    if isinstance(image, np.ndarray):
        arr = image
        if arr.dtype != np.uint8:
            arr = np.clip(arr, 0.0, 1.0)
            if arr.max() <= 1.0:
                arr = (arr * 255.0).astype(np.uint8)
            else:
                arr = arr.astype(np.uint8)
        if arr.ndim == 2:
            raise ValueError("grayscale numpy arrays are not supported; use RGB/RGBA")
        if arr.shape[-1] == 4:
            return Image.fromarray(arr, mode="RGBA")
        if arr.shape[-1] == 3:
            return Image.fromarray(arr, mode="RGB").convert("RGBA")
        raise ValueError(f"expected HxWx3 or HxWx4 array, got shape {arr.shape}")
    raise TypeError(f"unsupported image type: {type(image)}")

class TossInference:
    """Load TOSS once, then call ``generate`` with varying inputs (notebook-friendly)."""

    def __init__(
        self,
        model_cfg: str | Path = "models/toss_vae.yaml",
        resume_path: str | Path = "/content/drive/MyDrive/checkpoints/toss.ckpt",
        *,
        device: torch.device | str | None = None,
        gpu: int = 0,
        register_scheduler: bool = False,
        lr: float = 1e-4,
        seed: int = 40,
        sd_locked: bool = True,
        only_mid_control: bool = False,
        use_ema_scope: bool = True,
        pose_enc: str = "freq",
        h: int = 256,
        w: int = 256,
    ):
        """
        Args:
            model_cfg: YAML defining the model (e.g. ``models/toss_vae.yaml``).
            resume_path: Checkpoint with ``state_dict`` (e.g. ``ckpt/toss.ckpt``).
            device: Explicit device; if None, uses ``cuda:{gpu}`` when available.
            gpu: CUDA index when ``device`` is None.
            register_scheduler: Passed through to ``load_model`` (same as training flag).
            lr: Required by ``load_model``; not used during inference.
            seed: Fixed at init; call ``set_seed`` to change between runs.
            use_ema_scope / pose_enc / h / w: Defaults for ``generate`` (overridable per call).
        """
        seed_everything(seed, workers=True)
        if device is None:
            self.device = torch.device(
                f"cuda:{gpu}" if torch.cuda.is_available() else "cpu"
            )
        else:
            self.device = torch.device(device)

        hparams = SimpleNamespace(
            resume_path=str(resume_path),
            register_scheduler=register_scheduler,
            lr=lr,
        )
        cfgs = OmegaConf.load(str(model_cfg))
        self.model = load_model(
            self.device, hparams, sd_locked, only_mid_control, cfgs
        )

        trainable = [n for n, p in self.model.named_parameters() if p.requires_grad]
        print("Trainable parameters:", len(trainable))
        for n in trainable:
            print(n)

        self.sampler = DDIMSampler(self.model)

        self._default_use_ema_scope = use_ema_scope
        self._default_pose_enc = pose_enc
        self._default_h = h
        self._default_w = w

    def set_seed(self, seed: int) -> None:
        """Call between ``generate`` runs for reproducible DDIM noise."""
        seed_everything(seed, workers=True)

    @torch.no_grad()
    def generate(
        self,
        image: ImageInput,
        prompt: str = "",
        dx: float = 0.0,
        dy: float = 0.0,
        dz: float = 0.0,
        *,
        pose_enc: str | None = None,
        h: int | None = None,
        w: int | None = None,
        precision: str = "fp32",
        n_samples: int = 1,
        use_ema_scope: bool | None = None,
        ddim_steps: int = 100,
        ddim_eta: float = 1.0,
        prompt_scale: float = 5.0,
        img_scale: float = 3.0,
        img_ucg: float = 0.05,
    ) -> Image.Image:
        """
        Novel view for one image (same logic as ``app.generate_loop_views`` / ``sample_model``).

        Args:
            image: Path, ``PIL.Image``, or ``HxWx3`` / ``HxWx4`` numpy array (float or uint8).
            prompt: Text conditioning (empty string allowed).
            dx, dy, dz: Relative pose in degrees / distance (see ``app.get_T_from_relative``).
        """
        h = self._default_h if h is None else h
        w = self._default_w if w is None else w
        pose_enc = self._default_pose_enc if pose_enc is None else pose_enc
        if use_ema_scope is None:
            use_ema_scope = self._default_use_ema_scope

        cond_im_pil = _to_pil_rgba(image)
        cond_im = preprocess_image(cond_im_pil)
        cond_im = transforms.ToTensor()(cond_im).unsqueeze(0).to(self.device)
        cond_im = transforms.functional.resize(cond_im, [h, w])

        T = get_T_from_relative(dx, dy, dz, pose_enc)
        x_samples = sample_model(
            cond_im,
            self.model,
            self.sampler,
            precision=precision,
            h=h,
            w=w,
            ddim_steps=ddim_steps,
            n_samples=n_samples,
            prompt_scale=prompt_scale,
            img_scale=img_scale,
            ddim_eta=ddim_eta,
            T=T,
            use_ema_scope=use_ema_scope,
            prompt=prompt,
            img_ucg=img_ucg,
        )
        assert x_samples.shape[0] == 1
        out = x_samples[0].cpu().numpy()
        out = 255.0 * rearrange(out, "c h w -> h w c")
        return Image.fromarray(out.astype(np.uint8))

In [ ]:
toss = TossInference()

In [ ]:
import numpy as np

def preprocess_image(input_im, fg_mask=None):
    '''
    :param input_im (PIL Image).
    :param fg_mask: optional PIL Image (L mode) foreground mask, e.g. 00008_mask.png.
        When provided, used as alpha for white-background compositing (preferred over RGBA alpha).
    :return input_im (H, W, 3) array in [0, 1].
    '''
    input_im = input_im.resize([256, 256], Image.Resampling.LANCZOS)
    rgb = np.asarray(input_im.convert("RGB"), dtype=np.float32) / 255.0

    alpha = None
    if fg_mask is not None:
        fg_mask = fg_mask.resize([256, 256], Image.Resampling.LANCZOS)
        alpha = np.asarray(fg_mask, dtype=np.float32) / 255.0
        if alpha.ndim == 3:
            alpha = alpha[..., 0]
        alpha = alpha[..., None]  # [H, W, 1]
    elif input_im.mode == "RGBA":
        rgba = np.asarray(input_im, dtype=np.float32) / 255.0
        rgb = rgba[:, :, :3]
        alpha = rgba[:, :, 3:4]

    if alpha is not None:
        white_im = np.ones_like(rgb)
        rgb = alpha * rgb + (1.0 - alpha) * white_im

    return rgb

def rotation_matrix_to_euler(R):
    """
    Extract pitch (x-rotation) and yaw (y-rotation) from a 3x3 rotation matrix.
    Returns angles in radians.

    Assumes rotation order: R = Ry(yaw) @ Rx(pitch) @ Rz(roll)
    """
    # Clamp to avoid numerical issues with asin
    sy = np.clip(R[0, 2], -1.0, 1.0)
    yaw = np.arcsin(sy)

    # Check for gimbal lock
    if np.abs(sy) < 0.99999:
        pitch = np.arctan2(-R[1, 2], R[2, 2])
    else:
        pitch = np.arctan2(R[2, 1], R[1, 1])

    return pitch, yaw


def pose_matrix_to_toss_format(pose_4x4):
    """
    Convert a 4x4 pose matrix to TOSS format: [pitch, yaw, distance]

    Args:
        pose_4x4: 4x4 transformation matrix (camera-to-world or world-to-camera)

    Returns:
        [pitch, yaw, distance] as expected by TOSS pose_enc="vae" or "freq"
    """
    R = pose_4x4[:3, :3]  # 3x3 rotation
    t = pose_4x4[:3, 3]   # translation vector

    pitch, yaw = rotation_matrix_to_euler(R)

    # Distance: typically the Z component or the norm of translation
    # Adjust based on your coordinate system
    distance = np.linalg.norm(t)  # or t[2] if Z is the depth axis

    return np.array([pitch, yaw, distance], dtype=np.float32)

def _wrap_angle(angle: float) -> float:
    return float(np.arctan2(np.sin(angle), np.cos(angle)))


def compute_relative_pose(
    src_pose_4x4: np.ndarray, tgt_pose_4x4: np.ndarray
) -> np.ndarray:
    """
    Relative [delta_pitch, delta_yaw, delta_distance] in radians.
    Nersemble yaw sign is flipped to match Portrait4D/TOSS convention.
    """
    src_pitch, src_yaw = rotation_matrix_to_euler(src_pose_4x4[:3, :3])
    tgt_pitch, tgt_yaw = rotation_matrix_to_euler(tgt_pose_4x4[:3, :3])

    delta_pitch = _wrap_angle(tgt_pitch - src_pitch)

    # Nersemble: left-looking image gives negative yaw,
    # Portrait4D/TOSS: left-looking image gives positive yaw.
    delta_yaw = -_wrap_angle(tgt_yaw - src_yaw)

    src_dist = float(np.linalg.norm(src_pose_4x4[:3, 3]))
    tgt_dist = float(np.linalg.norm(tgt_pose_4x4[:3, 3]))

    return np.array(
        [delta_pitch, delta_yaw, tgt_dist - src_dist],
        dtype=np.float32,
    )


# def compute_relative_pose(src_pose_4x4, tgt_pose_4x4):
#     """
#     Compute relative pose from source to target view.
#     Returns [delta_pitch, delta_yaw, delta_distance] in radians.
#     """
#     src_pitch, src_yaw = rotation_matrix_to_euler(src_pose_4x4[:3, :3])
#     tgt_pitch, tgt_yaw = rotation_matrix_to_euler(tgt_pose_4x4[:3, :3])

#     src_dist = np.linalg.norm(src_pose_4x4[:3, 3])
#     tgt_dist = np.linalg.norm(tgt_pose_4x4[:3, 3])

#     delta_pitch = tgt_pitch - src_pitch
#     delta_yaw = tgt_yaw - src_yaw
#     delta_distance = tgt_dist - src_dist

#     return np.array([delta_pitch, delta_yaw, delta_distance], dtype=np.float32)

# Debug: Check poses.npy format and values
import numpy as np

sub_path = "/content/drive/MyDrive/datasets/TOSS_Dataset/portrait4d_rembg_isnet_v2/202501"
# sub_path = "/content/drive/MyDrive/datasets/TOSS_Dataset/nersemble_v2/017"
poses = np.load(f"{sub_path}/poses.npy")

print(f"Poses shape: {poses.shape}")
print(f"Poses dtype: {poses.dtype}")

# Reshape if needed
if poses.ndim == 2 and poses.shape[1] == 16:
    poses = poses.reshape(-1, 4, 4)
    print(f"Reshaped to: {poses.shape}")

src_idx = 0

print("\nRelative pose values:")
for i in range(len(poses)):
    rel = compute_relative_pose(poses[src_idx], poses[i])
    print(
        f"View {i:02d}: "
        f"d_pitch={np.degrees(rel[0]):+7.2f}°, "
        f"d_yaw={np.degrees(rel[1]):+7.2f}°, "
        f"d_dist={rel[2]:+.4f}"
    )

# Check a single pose matrix
print(f"\nPose[0] (view 00000):\n{poses[0]}")
print(f"\nPose[9] (view 00009 - center):\n{poses[9]}")

# Check if it looks like a valid rotation matrix
R = poses[0][:3, :3]
print(f"\nRotation matrix R (from pose[0]):\n{R}")
print(f"det(R) = {np.linalg.det(R):.6f} (should be ~1.0 for valid rotation)")
print(f"R @ R.T =\n{R @ R.T} (should be ~identity)")

# Extract angles using your function
pitch, yaw = rotation_matrix_to_euler(R)
print(f"\nExtracted angles from pose[0]:")
print(f"  pitch = {pitch:.4f} rad = {np.degrees(pitch):.2f}°")
print(f"  yaw = {yaw:.4f} rad = {np.degrees(yaw):.2f}°")

# Check all poses' yaw values
print(f"\nAll yaw values (degrees):")
for i in range(min(20, len(poses))):
    p, y = rotation_matrix_to_euler(poses[i][:3, :3])
    print(f"  View {i:02d}: pitch={np.degrees(p):+7.2f}°, yaw={np.degrees(y):+7.2f}°")

In [ ]:
# normal mask edit
import os
from PIL import Image
from torch.utils.data import Dataset
import numpy as np
import torch
from torchvision import transforms as T

import os

def preprocess_normal_for_cosine(normal_rgb, fg_mask=None, bg_color=None, bg_eps=0.02):
    """
    Preprocess normal map for cosine similarity loss.

    The normal PNG has two invalid regions:
      (1) uncropping padding outside the pixel3dmm bbox (constant color)
      (2) around-head pixels inside the bbox that pixel3dmm wrote garbage into
          (excluded via the FG/rembg silhouette mask)

    Args:
        normal_rgb: [H, W, 3] or [3, H, W], values in [0, 1] (standard normal map: RGB = xyz)
        fg_mask: optional [1, H, W] or [H, W] foreground (rembg) mask in [0, 1].
            If provided, the validity mask is intersected with it, removing
            inside-bbox-but-not-head pixels (shoulders, neck cutoff, in-crop bg, etc).
        bg_color: optional [3] iterable in [0, 1] specifying the uncropping padding color.
            If None, it is auto-detected as the median of the four corner pixels.
        bg_eps: max L_inf distance (in [0,1] RGB space) from bg_color to count as padding.

    Returns:
        normal: [3, H, W] L2-normalized, in [-1, 1] range (zeroed where invalid)
        valid_mask: [1, H, W] float, 1 where valid (head region), 0 elsewhere
    """
    if isinstance(normal_rgb, np.ndarray):
        normal = torch.from_numpy(normal_rgb).float()
    else:
        normal = normal_rgb.float()

    if normal.ndim == 3 and normal.shape[0] != 3 and normal.shape[-1] == 3:
        normal = normal.permute(2, 0, 1)  # HWC -> CHW

    # (1) Padding mask: detect padding color from four corners (median is robust to
    # one corner accidentally containing face).
    if bg_color is None:
        corners = torch.stack([
            normal[:, 0, 0],
            normal[:, 0, -1],
            normal[:, -1, 0],
            normal[:, -1, -1],
        ], dim=0)  # [4, 3]
        bg_color = corners.median(dim=0).values  # [3]
    else:
        bg_color = torch.as_tensor(bg_color, dtype=normal.dtype)

    diff = (normal - bg_color.view(3, 1, 1)).abs().amax(dim=0, keepdim=True)  # [1,H,W]
    in_bbox_mask = (diff > bg_eps).float()  # 1 inside the pixel3dmm crop, 0 in padding

    # (2) FG mask: removes inside-bbox-but-not-head pixels
    if fg_mask is not None:
        if isinstance(fg_mask, np.ndarray):
            fg_mask = torch.from_numpy(fg_mask).float()
        fg_mask = fg_mask.float()
        if fg_mask.ndim == 2:
            fg_mask = fg_mask.unsqueeze(0)  # [1, H, W]
        fg_bin = (fg_mask > 0.5).float()
        valid_mask = in_bbox_mask * fg_bin
    else:
        valid_mask = in_bbox_mask

    # RGB [0, 1] -> xyz [-1, 1], then L2-normalize per pixel
    normal = normal * 2.0 - 1.0
    normal = normal / normal.norm(dim=0, keepdim=True).clamp(min=1e-8)
    normal = normal * valid_mask  # zero out invalid pixels

    return normal, valid_mask

def preprocess_image(input_im, fg_mask=None):
    '''
    :param input_im (PIL Image).
    :param fg_mask: optional PIL Image (L mode) foreground mask, e.g. 00008_mask.png.
        When provided, used as alpha for white-background compositing (preferred over RGBA alpha).
    :return input_im (H, W, 3) array in [0, 1].
    '''
    input_im = input_im.resize([256, 256], Image.Resampling.LANCZOS)
    rgb = np.asarray(input_im.convert("RGB"), dtype=np.float32) / 255.0

    alpha = None
    if fg_mask is not None:
        fg_mask = fg_mask.resize([256, 256], Image.Resampling.LANCZOS)
        alpha = np.asarray(fg_mask, dtype=np.float32) / 255.0
        if alpha.ndim == 3:
            alpha = alpha[..., 0]
        alpha = alpha[..., None]  # [H, W, 1]
    elif input_im.mode == "RGBA":
        rgba = np.asarray(input_im, dtype=np.float32) / 255.0
        rgb = rgba[:, :, :3]
        alpha = rgba[:, :, 3:4]

    if alpha is not None:
        white_im = np.ones_like(rgb)
        rgb = alpha * rgb + (1.0 - alpha) * white_im

    return rgb

def _normal_to_rgb_vis(norm):
    """Map unit normal field to RGB in [0,1] for visualization. norm: [B,3,H,W] or [3,H,W]. Returns [3,H,W]."""
    if norm.dim() == 4:
        norm = norm[0]
    return torch.clamp((norm + 1) / 2, 0, 1)

class Portrait4dDataset(Dataset):
    def __init__(self, root_dir, transform=None, src_view_idx=3, skip_identity=True,
                 load_normals=False, load_depth=False, load_corr=False):
        super().__init__()
        self.root_dir = root_dir
        self.transform = transform
        self.src_view_idx = src_view_idx
        self.skip_identity = skip_identity
        self.load_normals = load_normals
        self.load_depth = load_depth
        self.load_corr = load_corr

        # Per-subject stacked correlation supervision (indexed by view_index):
        #   correlation.npy   [num_views, Hf, Wf]      -> corr_gt
        #   valid.npy         [num_views, Hf, Wf]      -> valid_mask
        #   flow_tgt2src.npy  [num_views, Hf, Wf, 2]   -> flow_tgt2src (grid_sample grid)
        self.corr_file = "correlation.npy"
        self.valid_file = "valid.npy"
        self.flow_file = "flow_tgt2src.npy"

        self.subjects = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.samples = []
        skipped = 0

        for sub in self.subjects:
            sub_path = os.path.join(root_dir, sub)
            poses_path = os.path.join(sub_path, 'poses.npy')
            if not os.path.exists(poses_path):
                skipped += 1
                continue

            views = sorted([
                f for f in os.listdir(sub_path)
                if os.path.isfile(os.path.join(sub_path, f))
                and os.path.splitext(f)[1].lower() in [".jpg", ".png"]
                and os.path.splitext(f)[0].isdigit()
            ])
            # views = sorted([f for f in os.listdir(sub_path)
            #                if f.endswith('.jpg') and not f.startswith('src')])

            # Correlation arrays are stacked over all NON-source views in sorted order,
            # so map each target view_index to its row (the source view is excluded).
            corr_index_map = {}
            if self.load_corr:
                non_src_sorted = [
                    int(os.path.splitext(f)[0]) for f in views
                    if int(os.path.splitext(f)[0]) != self.src_view_idx
                ]
                corr_index_map = {v: i for i, v in enumerate(non_src_sorted)}

            for view_file in views:
                view_id = view_file.split('.')[0]
                view_index = int(view_id)
                if self.skip_identity and view_index == self.src_view_idx:
                    continue

                mask_file = f"{view_id}_mask.png"
                normal_file = os.path.join("normals_uncropped", f"{view_id}.png")
                normal_path = os.path.join(sub_path, normal_file)
                depth_file = os.path.join("depth", f"{view_id}.npy")
                depth_path = os.path.join(sub_path, depth_file)

                has_mask = os.path.exists(os.path.join(sub_path, mask_file))
                has_normal = os.path.exists(normal_path) if self.load_normals else True
                has_depth = os.path.exists(depth_path) if self.load_depth else True
                if self.load_corr:
                    has_corr = (
                        os.path.exists(os.path.join(sub_path, self.corr_file))
                        and os.path.exists(os.path.join(sub_path, self.valid_file))
                        and os.path.exists(os.path.join(sub_path, self.flow_file))
                    )
                else:
                    has_corr = True

                if has_mask and has_normal and has_depth and has_corr:
                    self.samples.append({
                        'sub_path': sub_path,
                        'view_file': view_file,
                        'mask_file': mask_file,
                        'normal_file': normal_file,
                        'depth_file': depth_file,
                        'view_index': view_index,
                        'corr_index': corr_index_map.get(view_index),
                    })

        parts = []
        if self.load_normals:
            parts.append("normals")
        if self.load_depth:
            parts.append("depth")
        if self.load_corr:
            parts.append("correlation")
        extra_str = f" (with {', '.join(parts)})" if parts else ""
        print(f"[Dataset] Loaded {len(self.samples)} samples{extra_str}, skipped {skipped} incomplete subjects")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        sub_path = sample['sub_path']

        src_path = os.path.join(sub_path, f"{self.src_view_idx:05d}.png")
        src = Image.open(src_path).convert("RGBA")
        src_mask_path = os.path.join(sub_path, f"{self.src_view_idx:05d}_mask.png")
        src_mask = Image.open(src_mask_path).convert("L")

        img_path = os.path.join(sub_path, sample['view_file'])
        image = Image.open(img_path).convert("RGBA")

        mask_path = os.path.join(sub_path, sample['mask_file'])
        mask = Image.open(mask_path).convert("L")

        poses = np.load(os.path.join(sub_path, 'poses.npy'))
        if poses.ndim == 2 and poses.shape[1] == 16:
            poses = poses.reshape(-1, 4, 4)

        delta_pose = compute_relative_pose(
            poses[self.src_view_idx],
            poses[sample['view_index']]
        )
        assert abs(delta_pose[1]) < 1.0, f"Yaw {delta_pose[1]} seems too large - check units!"
        delta_pose = torch.from_numpy(delta_pose).float()

        src = preprocess_image(src, fg_mask=src_mask)       # Returns [H, W, 3] numpy array
        image = preprocess_image(image, fg_mask=mask)       # Returns [H, W, 3] numpy array

        src = torch.from_numpy(src).permute(2, 0, 1).float()
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        if self.transform:
            src = self.transform(src)
            image = self.transform(image)

        mask_transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
        mask = mask_transform(mask)

        out = {
            "jpg": image,
            "hint": src,
            "mask": mask,
            "delta_pose": delta_pose,
            "subject_id": os.path.basename(sub_path),
            "txt": ""
        }

        if self.load_normals:
            normal_path = os.path.join(sub_path, sample['normal_file'])
            normal_img = Image.open(normal_path).convert("RGB")
            # NEAREST avoids bleeding padding color into face pixels along the boundary
            normal_img = normal_img.resize((256, 256), Image.NEAREST)
            normal_np = np.array(normal_img).astype(np.float32) / 255.0
            # Two-stage masking:
            #   (1) padding-color detection removes the uncropped boundary
            #   (2) FG (rembg) mask removes inside-bbox-but-not-head pixels
            fg_for_normal = mask[0].numpy()  # [H, W] in [0, 1]
            normal, normal_valid_mask = preprocess_normal_for_cosine(
                normal_np, fg_mask=fg_for_normal
            )
            out["normal"] = normal
            out["normal_mask"] = normal_valid_mask

        if self.load_depth:
            depth_path = os.path.join(sub_path, sample['depth_file'])
            depth_np = np.load(depth_path).astype(np.float32)
            if depth_np.ndim == 3:  # (H, W, 1) or (1, H, W) -> (H, W)
                depth_np = depth_np.squeeze()
            # Sanitize BEFORE interpolation: bilinear weights spread NaN to neighbors
            depth_np = np.nan_to_num(depth_np, nan=0.0, posinf=0.0, neginf=0.0)
            depth_t = torch.from_numpy(depth_np).float().unsqueeze(0)  # [1, H, W]
            depth_t = torch.nn.functional.interpolate(
                depth_t.unsqueeze(0), size=(256, 256),
                mode='bilinear', align_corners=False
            ).squeeze(0)  # [1, H, W]
            depth_t = torch.clamp(depth_t, min=0.0)  # ensure non-negative
            valid = (depth_t > 1e-6) & (depth_t < 1e6)
            depth_mask = valid.float()
            out["depth"] = depth_t
            out["depth_mask"] = depth_mask

        if self.load_corr:
            # Correlation arrays exclude the source view, so index by the precomputed
            # position among sorted non-source views (NOT the raw view_index).
            ci = sample['corr_index']
            assert ci is not None, (
                f"No correlation row for view_index {sample['view_index']} in {sub_path} "
                f"(is it the source view?)"
            )
            corr_arr = np.load(os.path.join(sub_path, self.corr_file))    # [num_tgt_views, Hf, Wf]
            valid_arr = np.load(os.path.join(sub_path, self.valid_file))  # [num_tgt_views, Hf, Wf]
            flow_arr = np.load(os.path.join(sub_path, self.flow_file))    # [num_tgt_views, Hf, Wf, 2]
            assert 0 <= ci < corr_arr.shape[0], (
                f"corr_index {ci} (view {sample['view_index']}) out of range for "
                f"correlation array {corr_arr.shape} in {sub_path}"
            )
            # corr_gt / valid_mask: [1, Hf, Wf]; flow_tgt2src: [Hf, Wf, 2] (already normalized to [-1, 1])
            out["corr_gt"] = torch.from_numpy(corr_arr[ci]).float().unsqueeze(0)
            out["valid_mask"] = torch.from_numpy(valid_arr[ci].astype(np.float32)).unsqueeze(0)
            out["flow_tgt2src"] = torch.from_numpy(flow_arr[ci]).float()

        return out

from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader

transform = T.Compose([
    T.Resize((256, 256))
])

DATASET_ROOT = "/content/drive/MyDrive/datasets/TOSS_Dataset/nersemble_processed_v2"
TRAIN_ROOT = os.path.join(DATASET_ROOT, "train")
TEST_ROOT = os.path.join(DATASET_ROOT, "test")
# dataset_name = "portrait4d_rembg_isnet"

train_dataset = Portrait4dDataset(root_dir=TRAIN_ROOT, transform=transform)
test_dataset  = Portrait4dDataset(root_dir=TEST_ROOT, transform=transform)
print(f"train identities ({len(train_dataset.subjects)}):", train_dataset.subjects)
print(f"test  identities ({len(test_dataset.subjects)}):", test_dataset.subjects)

dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)

data_iter = iter(dataloader)
batch = next(data_iter)
print("Batch keys:", batch.keys())

In [ ]:
import os
import numpy as np
import torch
from PIL import Image


def compute_psnr(pred, target, mask=None, data_range=1.0, eps=1e-10):
    """
    pred, target:
        PIL Image, numpy array, 또는 torch.Tensor
        최종 shape은 (H, W, 3), 값 범위는 [0, 1]

    mask:
        None 또는 (H, W), (H, W, 1)
        1인 영역에서만 PSNR 계산
    """

    def to_numpy_rgb(x):
        if isinstance(x, Image.Image):
            x = np.asarray(x.convert("RGB"), dtype=np.float32) / 255.0

        elif torch.is_tensor(x):
            x = x.detach().cpu().float().numpy()

            # CHW -> HWC
            if x.ndim == 3 and x.shape[0] in (1, 3, 4):
                x = np.transpose(x, (1, 2, 0))

        else:
            x = np.asarray(x, dtype=np.float32)

        # RGBA -> RGB
        if x.ndim == 3 and x.shape[-1] == 4:
            x = x[..., :3]

        # grayscale -> RGB
        if x.ndim == 2:
            x = np.repeat(x[..., None], 3, axis=-1)

        # uint8 또는 [0, 255] 실수 배열 처리
        if x.max() > 1.0:
            x = x / 255.0

        return np.clip(x.astype(np.float32), 0.0, 1.0)

    pred = to_numpy_rgb(pred)
    target = to_numpy_rgb(target)

    if pred.shape != target.shape:
        raise ValueError(
            f"pred와 target shape이 다릅니다: "
            f"pred={pred.shape}, target={target.shape}"
        )

    squared_error = (pred - target) ** 2

    if mask is not None:
        if isinstance(mask, Image.Image):
            mask = np.asarray(mask.convert("L"), dtype=np.float32) / 255.0
        elif torch.is_tensor(mask):
            mask = mask.detach().cpu().float().numpy()
        else:
            mask = np.asarray(mask, dtype=np.float32)

        mask = np.squeeze(mask)

        if mask.max() > 1.0:
            mask = mask / 255.0

        if mask.shape != pred.shape[:2]:
            raise ValueError(
                f"mask 크기가 이미지와 다릅니다: "
                f"mask={mask.shape}, image={pred.shape[:2]}"
            )

        valid = mask > 0.5

        if not np.any(valid):
            return float("nan")

        # valid pixel의 RGB 채널 전체에 대해 평균
        mse = float(np.mean(squared_error[valid]))

    else:
        mse = float(np.mean(squared_error))

    if mse <= eps:
        return float("inf")

    return float(
        20.0 * np.log10(data_range)
        - 10.0 * np.log10(mse)
    )

In [ ]:
import lpips
import torch
import torch.nn.functional as F
from PIL import Image


lpips_model = lpips.LPIPS(net="vgg", spatial=True).to(toss.device).eval()


def compute_lpips(pred, target, mask=None, lpips_model=lpips_model, device=None):
    """
    pred, target:
        PIL Image, numpy array, 또는 torch.Tensor
        최종 shape은 (H, W, 3), 값 범위는 [0, 1]

    mask:
        None 또는 (H, W), (H, W, 1)
        1인 영역에서만 LPIPS 평균 (spatial map 사용)

    returns:
        float (lower is better)
    """

    def to_numpy_rgb(x):
        if isinstance(x, Image.Image):
            x = np.asarray(x.convert("RGB"), dtype=np.float32) / 255.0

        elif torch.is_tensor(x):
            x = x.detach().cpu().float().numpy()

            # CHW -> HWC
            if x.ndim == 3 and x.shape[0] in (1, 3, 4):
                x = np.transpose(x, (1, 2, 0))

        else:
            x = np.asarray(x, dtype=np.float32)

        # RGBA -> RGB
        if x.ndim == 3 and x.shape[-1] == 4:
            x = x[..., :3]

        # grayscale -> RGB
        if x.ndim == 2:
            x = np.repeat(x[..., None], 3, axis=-1)

        # uint8 또는 [0, 255] 실수 배열 처리
        if x.max() > 1.0:
            x = x / 255.0

        return np.clip(x.astype(np.float32), 0.0, 1.0)

    def to_tensor_nchw_minus1_1(x):
        arr = to_numpy_rgb(x)
        tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
        return tensor * 2.0 - 1.0

    device = device or toss.device
    pred_t = to_tensor_nchw_minus1_1(pred).to(device)
    target_t = to_tensor_nchw_minus1_1(target).to(device)

    if pred_t.shape != target_t.shape:
        raise ValueError(
            f"pred와 target shape이 다릅니다: "
            f"pred={pred_t.shape}, target={target_t.shape}"
        )

    with torch.no_grad():
        dist_map = lpips_model(pred_t, target_t)

    if mask is None:
        return float(dist_map.mean().item())

    if isinstance(mask, Image.Image):
        mask = np.asarray(mask.convert("L"), dtype=np.float32) / 255.0
    elif torch.is_tensor(mask):
        mask = mask.detach().cpu().float().numpy()
    else:
        mask = np.asarray(mask, dtype=np.float32)

    mask = np.squeeze(mask)

    if mask.max() > 1.0:
        mask = mask / 255.0

    if mask.shape != pred_t.shape[-2:]:
        raise ValueError(
            f"mask 크기가 이미지와 다릅니다: "
            f"mask={mask.shape}, image={pred_t.shape[-2:]}"
        )

    mask_t = torch.from_numpy(mask).float().unsqueeze(0).unsqueeze(0).to(device)

    if mask_t.shape[-2:] != dist_map.shape[-2:]:
        mask_t = F.interpolate(
            mask_t,
            size=dist_map.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

    valid = mask_t > 0.5

    if not valid.any():
        return float("nan")

    return float(dist_map[valid].mean().item())

In [ ]:
import torch.nn.functional as F
from PIL import Image
from cldm.arcface_torch_wrapper import (
    create_frozen_arcface_backbone,
    preprocess_arcface_input,
)


ARCFACE_CKPT_PATH = "/content/drive/MyDrive/checkpoints/ms1mv3_arcface_r100_fp16.pth"
ARCFACE_SPATIAL_MODE = "cover_center"

arcface_backbone = create_frozen_arcface_backbone(ARCFACE_CKPT_PATH).to(toss.device).eval()


def compute_identity_similarity(
    pred,
    target,
    backbone=arcface_backbone,
    spatial_mode=ARCFACE_SPATIAL_MODE,
    device=None,
):
    """
    pred, target:
        PIL Image, numpy array, 또는 torch.Tensor
        최종 shape은 (H, W, 3), 값 범위는 [0, 1]

    returns:
        float cosine similarity in [-1, 1] (higher is better)
    """

    def to_numpy_rgb(x):
        if isinstance(x, Image.Image):
            x = np.asarray(x.convert("RGB"), dtype=np.float32) / 255.0

        elif torch.is_tensor(x):
            x = x.detach().cpu().float().numpy()

            # CHW -> HWC
            if x.ndim == 3 and x.shape[0] in (1, 3, 4):
                x = np.transpose(x, (1, 2, 0))

        else:
            x = np.asarray(x, dtype=np.float32)

        # RGBA -> RGB
        if x.ndim == 3 and x.shape[-1] == 4:
            x = x[..., :3]

        # grayscale -> RGB
        if x.ndim == 2:
            x = np.repeat(x[..., None], 3, axis=-1)

        # uint8 또는 [0, 255] 실수 배열 처리
        if x.max() > 1.0:
            x = x / 255.0

        return np.clip(x.astype(np.float32), 0.0, 1.0)

    def to_tensor_nchw_01(x):
        arr = to_numpy_rgb(x)
        return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)

    device = device or toss.device
    pred_t = to_tensor_nchw_01(pred).to(device)
    target_t = to_tensor_nchw_01(target).to(device)

    if pred_t.shape != target_t.shape:
        raise ValueError(
            f"pred와 target shape이 다릅니다: "
            f"pred={pred_t.shape}, target={target_t.shape}"
        )

    with torch.no_grad():
        pred_arc = preprocess_arcface_input(
            pred_t,
            spatial_mode=spatial_mode,
        )
        target_arc = preprocess_arcface_input(
            target_t,
            spatial_mode=spatial_mode,
        )
        emb_pred = F.normalize(backbone(pred_arc), dim=-1)
        emb_target = F.normalize(backbone(target_arc), dim=-1)

    return float((emb_pred * emb_target).sum(dim=-1).item())

In [ ]:
src_view_idx = 3
test_views = [12, 0, 15, 1]

test_subjects = test_dataset.subjects

per_view_psnr = {v: [] for v in test_views}
per_view_fg_psnr = {v: [] for v in test_views}
per_view_lpips = {v: [] for v in test_views}
per_view_fg_lpips = {v: [] for v in test_views}
per_view_id_sim = {v: [] for v in test_views}

per_subject_psnr = {str(s): [] for s in test_subjects}
per_subject_fg_psnr = {str(s): [] for s in test_subjects}
per_subject_lpips = {str(s): [] for s in test_subjects}
per_subject_fg_lpips = {str(s): [] for s in test_subjects}
per_subject_id_sim = {str(s): [] for s in test_subjects}

all_psnr = []
all_fg_psnr = []
all_lpips = []
all_fg_lpips = []
all_id_sim = []

for subject in test_subjects:
    subject = str(subject)
    sub_path = os.path.join(TEST_ROOT, subject)

    poses = np.load(
        os.path.join(sub_path, "poses.npy")
    ).reshape(-1, 4, 4)

    # Source image
    src_img = Image.open(
        os.path.join(sub_path, f"{src_view_idx:05d}.png")
    ).convert("RGBA")

    src_mask = Image.open(
        os.path.join(sub_path, f"{src_view_idx:05d}_mask.png")
    ).convert("L")

    # 학습 때와 동일하게 mask로 흰 배경 합성
    src_np = preprocess_image(
        src_img,
        fg_mask=src_mask,
    )

    src_input = Image.fromarray(
        np.clip(src_np * 255.0, 0, 255).astype(np.uint8)
    )

    for view_idx in test_views:
        delta = compute_relative_pose(
            poses[src_view_idx],
            poses[view_idx],
        )
        delta_yaw_deg = float(np.degrees(delta[1]))

        with torch.no_grad():
            gen_pil = toss.generate(
                image=src_input,
                prompt="",
                dy=delta_yaw_deg,
            )

        # GT image
        gt_img = Image.open(
            os.path.join(sub_path, f"{view_idx:05d}.png")
        ).convert("RGBA")

        gt_mask = Image.open(
            os.path.join(sub_path, f"{view_idx:05d}_mask.png")
        ).convert("L")

        gt_np = preprocess_image(
            gt_img,
            fg_mask=gt_mask,
        )

        gen_pil = gen_pil.convert("RGB").resize(
            (256, 256),
            Image.Resampling.LANCZOS,
        )
        gen_np = np.asarray(gen_pil, dtype=np.float32) / 255.0

        gt_mask_np = np.asarray(
            gt_mask.resize(
                (256, 256),
                Image.Resampling.LANCZOS,
            ),
            dtype=np.float32,
        ) / 255.0

        # 전체 이미지 PSNR
        psnr = compute_psnr(
            gen_np,
            gt_np,
        )

        # 전경 영역 PSNR
        fg_psnr = compute_psnr(
            gen_np,
            gt_np,
            mask=gt_mask_np,
        )

        # 전체 이미지 LPIPS
        lpips_val = compute_lpips(
            gen_np,
            gt_np,
        )

        # 전경 영역 LPIPS
        fg_lpips = compute_lpips(
            gen_np,
            gt_np,
            mask=gt_mask_np,
        )

        # Identity similarity (ArcFace cosine similarity)
        id_sim = compute_identity_similarity(
            gen_np,
            gt_np,
        )

        per_view_psnr[view_idx].append(psnr)
        per_view_fg_psnr[view_idx].append(fg_psnr)
        per_view_lpips[view_idx].append(lpips_val)
        per_view_fg_lpips[view_idx].append(fg_lpips)
        per_view_id_sim[view_idx].append(id_sim)

        per_subject_psnr[subject].append(psnr)
        per_subject_fg_psnr[subject].append(fg_psnr)
        per_subject_lpips[subject].append(lpips_val)
        per_subject_fg_lpips[subject].append(fg_lpips)
        per_subject_id_sim[subject].append(id_sim)

        all_psnr.append(psnr)
        all_fg_psnr.append(fg_psnr)
        all_lpips.append(lpips_val)
        all_fg_lpips.append(fg_lpips)
        all_id_sim.append(id_sim)

        print(
            f"subject {subject}, view {view_idx:02d}, "
            f"dyaw={delta_yaw_deg:+.1f}° | "
            f"Full PSNR={psnr:.3f} dB | "
            f"FG PSNR={fg_psnr:.3f} dB | "
            f"Full LPIPS={lpips_val:.4f} | "
            f"FG LPIPS={fg_lpips:.4f} | "
            f"IdSim={id_sim:.4f}"
        )


print("\n=== Per-view mean PSNR ===")
for view_idx in test_views:
    print(
        f"view {view_idx:02d}: "
        f"Full={np.mean(per_view_psnr[view_idx]):.3f} dB | "
        f"FG={np.mean(per_view_fg_psnr[view_idx]):.3f} dB "
        f"(n={len(per_view_psnr[view_idx])})"
    )


print("\n=== Per-subject mean PSNR ===")
for subject in test_subjects:
    subject = str(subject)

    if per_subject_psnr[subject]:
        print(
            f"{subject}: "
            f"Full={np.mean(per_subject_psnr[subject]):.3f} dB | "
            f"FG={np.mean(per_subject_fg_psnr[subject]):.3f} dB"
        )


print(
    f"\n=== Overall mean PSNR ===\n"
    f"Full image: {np.mean(all_psnr):.3f} dB\n"
    f"Foreground: {np.mean(all_fg_psnr):.3f} dB"
)


print("\n=== Per-view mean LPIPS ===")
for view_idx in test_views:
    print(
        f"view {view_idx:02d}: "
        f"Full={np.mean(per_view_lpips[view_idx]):.4f} | "
        f"FG={np.mean(per_view_fg_lpips[view_idx]):.4f} "
        f"(n={len(per_view_lpips[view_idx])})"
    )


print("\n=== Per-subject mean LPIPS ===")
for subject in test_subjects:
    subject = str(subject)

    if per_subject_lpips[subject]:
        print(
            f"{subject}: "
            f"Full={np.mean(per_subject_lpips[subject]):.4f} | "
            f"FG={np.mean(per_subject_fg_lpips[subject]):.4f}"
        )


print(
    f"\n=== Overall mean LPIPS ===\n"
    f"Full image: {np.mean(all_lpips):.4f}\n"
    f"Foreground: {np.mean(all_fg_lpips):.4f}"
)


print("\n=== Per-view mean Identity Similarity (higher is better) ===")
for view_idx in test_views:
    print(
        f"view {view_idx:02d}: "
        f"IdSim={np.mean(per_view_id_sim[view_idx]):.4f} "
        f"(n={len(per_view_id_sim[view_idx])})"
    )


print("\n=== Per-subject mean Identity Similarity (higher is better) ===")
for subject in test_subjects:
    subject = str(subject)

    if per_subject_id_sim[subject]:
        print(
            f"{subject}: "
            f"IdSim={np.mean(per_subject_id_sim[subject]):.4f}"
        )


print(
    f"\n=== Overall mean Identity Similarity (higher is better) ===\n"
    f"IdSim={np.mean(all_id_sim):.4f}"
)